# 🧪 Solutions: Protein Analysis

Complete solutions for Kodomo Exercise 05

## Complicated moments explained
These are common points where learners usually get stuck:
- Focus on why each step exists, not only the final answer.
- Compare intermediate outputs to your own attempt, not just final metrics.
- Small implementation differences can still be correct if assumptions match.


In [ ]:
from collections import Counter

## Exercise 1: Amino Acid Composition

In [ ]:
def amino_acid_composition(protein_sequence):
    """Calculate amino acid composition with counts and percentages."""
    seq = protein_sequence.upper()
    seq_length = len(seq)
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    
    composition = {}
    for aa in amino_acids:
        count = seq.count(aa)
        percentage = (count / seq_length * 100) if seq_length > 0 else 0
        composition[aa] = {'count': count, 'percentage': round(percentage, 2)}
    
    return composition

# Test with insulin sequence
insulin = "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKT"
comp = amino_acid_composition(insulin)
print("Insulin amino acid composition:")
for aa, data in sorted(comp.items(), key=lambda x: -x[1]['count']):
    if data['count'] > 0:
        print(f"  {aa}: {data['count']} ({data['percentage']}%)")

## Exercise 2: Codon Table and Translation

In [ ]:
CODON_TABLE = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
}

def translate_dna(dna_sequence):
    """Translate DNA sequence to protein."""
    dna = dna_sequence.upper().replace('U', 'T')
    protein = []
    
    for i in range(0, len(dna) - 2, 3):
        codon = dna[i:i+3]
        aa = CODON_TABLE.get(codon, 'X')
        if aa == '*':
            break
        protein.append(aa)
    
    return ''.join(protein)

# Test
dna = "ATGGCTCTCTGGAAGCTTCTGGACGTCCTGCTT"
print(f"DNA: {dna}")
print(f"Protein: {translate_dna(dna)}")

## Exercise 3: Protein Properties

In [ ]:
# Amino acid properties
AA_MW = {  # Molecular weights
    'A': 89.1, 'C': 121.2, 'D': 133.1, 'E': 147.1, 'F': 165.2,
    'G': 75.1, 'H': 155.2, 'I': 131.2, 'K': 146.2, 'L': 131.2,
    'M': 149.2, 'N': 132.1, 'P': 115.1, 'Q': 146.2, 'R': 174.2,
    'S': 105.1, 'T': 119.1, 'V': 117.1, 'W': 204.2, 'Y': 181.2
}

HYDROPHOBIC = set('AVILMFYW')
POLAR = set('STNQCG')
CHARGED_POS = set('KRH')
CHARGED_NEG = set('DE')

def calculate_mw(sequence):
    """Calculate molecular weight of protein."""
    seq = sequence.upper()
    mw = sum(AA_MW.get(aa, 0) for aa in seq)
    mw -= (len(seq) - 1) * 18.015  # Water loss from peptide bonds
    return mw

def calculate_charge(sequence, ph=7.0):
    """Estimate net charge at given pH (simplified)."""
    seq = sequence.upper()
    pos = sum(1 for aa in seq if aa in CHARGED_POS)
    neg = sum(1 for aa in seq if aa in CHARGED_NEG)
    return pos - neg

def hydrophobicity_ratio(sequence):
    """Calculate percentage of hydrophobic residues."""
    seq = sequence.upper()
    hydro = sum(1 for aa in seq if aa in HYDROPHOBIC)
    return (hydro / len(seq) * 100) if len(seq) > 0 else 0

# Test
protein = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"
print(f"Protein: {protein[:30]}...")
print(f"Molecular weight: {calculate_mw(protein):.2f} Da")
print(f"Net charge (pH 7): {calculate_charge(protein)}")
print(f"Hydrophobicity: {hydrophobicity_ratio(protein):.1f}%")

## Exercise 4: Find ORFs

In [ ]:
def find_orfs(dna_sequence, min_length=30):
    """
    Find all Open Reading Frames in DNA sequence.
    Returns list of (start, end, protein) tuples.
    """
    dna = dna_sequence.upper()
    orfs = []
    
    for frame in range(3):
        i = frame
        while i < len(dna) - 2:
            codon = dna[i:i+3]
            if codon == 'ATG':  # Start codon
                start = i
                protein = ['M']
                j = i + 3
                
                while j < len(dna) - 2:
                    codon = dna[j:j+3]
                    aa = CODON_TABLE.get(codon, 'X')
                    if aa == '*':  # Stop codon
                        if len(protein) * 3 >= min_length:
                            orfs.append((start, j+3, ''.join(protein)))
                        break
                    protein.append(aa)
                    j += 3
                i = j + 3
            else:
                i += 3
    
    return orfs

# Test
test_dna = "ATGAAACCCGGGTTTTAAATGCCCAAATAG"
orfs = find_orfs(test_dna, min_length=6)
print(f"ORFs found in {test_dna}:")
for start, end, protein in orfs:
    print(f"  {start}-{end}: {protein}")

## Exercise 5: Amino Acid Classification

In [ ]:
def classify_amino_acids(sequence):
    """Classify amino acids in a protein sequence."""
    seq = sequence.upper()
    
    classification = {
        'hydrophobic': sum(1 for aa in seq if aa in HYDROPHOBIC),
        'polar': sum(1 for aa in seq if aa in POLAR),
        'positive': sum(1 for aa in seq if aa in CHARGED_POS),
        'negative': sum(1 for aa in seq if aa in CHARGED_NEG),
        'proline': seq.count('P')  # Special structure-breaker
    }
    
    # Add percentages
    total = len(seq)
    for key in classification:
        classification[key] = {
            'count': classification[key],
            'percentage': round(classification[key] / total * 100, 1)
        }
    
    return classification

# Test
protein = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"
classes = classify_amino_acids(protein)
print("Amino acid classification:")
for category, data in classes.items():
    print(f"  {category}: {data['count']} ({data['percentage']}%)")